# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a [Croissant schema URL](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).


In [ ]:
# Ensure `mlcroissant` is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Display dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview

Review the available record sets, fields, and their `@id`s in the dataset.

In [ ]:
# List available record sets and fields using their @id
print("Available record sets and their fields:")
record_set_ids = []
for record_set in metadata.record_sets:
    print(f"- RecordSet @id: {record_set.id}")
    record_set_ids.append(record_set.id)
    print(f"  Name: {record_set.name if hasattr(record_set, 'name') else ''}")
    print("  Fields:")
    for field in getattr(record_set, 'fields', []):
        print(f"    - Field @id: {field.id} (name: {field.name if hasattr(field, 'name') else ''})")
    # Optionally, print column details if present
    if hasattr(record_set, 'columns'):
        print("  Columns:")
        for column in record_set.columns:
            print(f"    - Column @id: {column.id} (name: {column.name if hasattr(column, 'name') else ''})")
    print()

## 3. Data Extraction

Extract data from record sets into pandas DataFrames. All identifiers should be referenced by their `@id`.

In [ ]:
# If the dataset contains record sets, load them
record_sets = record_set_ids  # Obtained from the previous cell

dataframes = {}
for record_set_id in record_sets:
    # Load all records for the record set
    print(f"Loading records from record set: {record_set_id}")
    try:
        records = list(dataset.records(record_set=record_set_id))
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for {record_set_id}.")
            print(f"Fields (@id): {df.columns.tolist()}")
            display(df.head())
        else:
            print(f"No records found for {record_set_id}.")
    except Exception as e:
        print(f"Error loading {record_set_id}: {e}")
print("\nRecord sets loaded:", list(dataframes.keys()))
if dataframes:
    # Select first available record set for further analysis
    main_record_set_id = list(dataframes.keys())[0]
    print(f"Proceeding with record set: {main_record_set_id}")
else:
    main_record_set_id = None


## 4. Exploratory Data Analysis (EDA)

Apply exploratory analysis on the data: filtering, normalization, grouping, etc., referencing fields using their `@id`.

In [ ]:
if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Columns in {main_record_set_id}: {df.columns.tolist()}")
    # Try to find a numeric field (by simple heuristic: dtype and/or name)
    numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if not numeric_fields:
        # Try to coerce columns to numeric
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col], errors='coerce')
                if converted.notna().sum() > 0:
                    numeric_fields.append(col)
            except Exception:
                continue
    print(f"Detected numeric fields: {numeric_fields}")
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        # Set threshold as mean + (std * 0.1) for the field
        field_data = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = field_data.mean() + 0.1 * field_data.std()
        filtered_df = df[field_data > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize this numeric field
        filtered_df[f"{numeric_field_id}_normalized"] = (field_data - field_data.mean()) / field_data.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
        
        # Try grouping by a likely categorical field
        group_fields = [col for col in df.columns if col != numeric_field_id]
        if group_fields:
            group_field = group_fields[0]
            try:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
                print(f"Grouped data by {group_field} (showing means for {numeric_field_id}):")
                display(grouped_df.head())
            except Exception as e:
                print(f"Could not group by {group_field}: {e}")
        else:
            print("No groupable field found.")
    else:
        print("No numeric fields detected for EDA.")
else:
    print("No data available for EDA.")

## 5. Visualization

Visualize distributions and relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and main_record_set_id in dataframes and numeric_fields:
    df = dataframes[main_record_set_id]
    numeric_field_id = numeric_fields[0]
    plt.figure(figsize=(8, 4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()
    # Optionally, scatter vs group_field
    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f'{numeric_field_id} by {group_field}')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No data available to plot.")

## 6. Conclusion

In this notebook, we loaded the FAIR^2 dataset using the Croissant schema with `mlcroissant`, explored its available record sets and fields using their `@id`s, extracted and previewed records into pandas DataFrames, applied basic exploratory data analysis, and visualized key numeric fields. This workflow enables reproducible FAIR-style exploration for research and analysis of complex data packages.